# 用 Sciverse 构建科研文献综述 Agent

> 从一句研究问题出发，自动检索、摘要、生成带引用的文献综述

**场景**: 科研人员或 AI Agent 需要针对一个研究问题，自动检索相关文献、提取关键证据段落，并生成一份带引用的文献综述。

**使用接口**: agentic-search, content

**预估调用量**: ~15–30 次 API 调用 / 一次综述任务

---


## Step 1: 环境准备

安装依赖并配置 API Token


In [ ]:
# 安装所需 Python 包
!pip install httpx anthropic
# 设置环境变量（替换为你的真实 Token）
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值
import os
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # 替换为你的真实值


## Step 2: 语义检索相关片段

使用 agentic-search 获取与研究问题最相关的文献片段


In [ ]:
import os
import asyncio
import httpx

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def search_literature(query: str, top_k: int = 20):
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{BASE}/agentic-search",
            headers=HEADERS,
            json={"query": query, "top_k": top_k}
        )
        resp.raise_for_status()
        return resp.json()["hits"]

async def main():
    hits = await search_literature(
        "Transformer applications in protein structure prediction 2020-2024"
    )
    print(f"Found {len(hits)} relevant chunks")
    for h in hits[:3]:
        print(f"  [{h['score']:.2f}] {h['title'][:60]}...")
    return hits

hits = asyncio.run(main())


## Step 3: 读取原文上下文

对高分片段调用 content 接口获取更完整的上下文


In [ ]:
async def read_context(doc_id: str, offset: int = 0, limit: int = 2000):
    """读取指定文档的原文片段。返回 {text, next_offset, more}"""
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(
            f"{BASE}/content",
            headers=HEADERS,
            params={"doc_id": doc_id, "offset": offset, "limit": limit}
        )
        resp.raise_for_status()
        return resp.json()

async def gather_evidence(hits, top_n=5):
    sorted_hits = sorted(hits, key=lambda x: x["score"], reverse=True)[:top_n]
    evidences = []
    for hit in sorted_hits:
        ctx = await read_context(hit["doc_id"], hit.get("offset", 0))
        evidences.append({
            "title": hit["title"],
            "doc_id": hit["doc_id"],
            "offset": hit.get("offset", 0),
            "chunk": hit["chunk"],
            "context": ctx["text"],  # 注意：响应字段是 text
            "score": hit["score"]
        })
    return evidences

evidences = asyncio.run(gather_evidence(hits))


## Step 4: 生成带引用的综述（可选增强）

将证据传给 LLM 生成结构化综述。此步骤依赖 Anthropic API Key，非 Sciverse 必需


In [ ]:
from anthropic import Anthropic

client = Anthropic()  # 自动读取 ANTHROPIC_API_KEY

evidence_text = "\
\
".join([
    f"[{e['doc_id']}, offset={e['offset']}] {e['title']}\
{e['context']}"
    for e in evidences
])

msg = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=4096,
    messages=[{
        "role": "user",
        "content": f"""基于以下文献证据，生成一份关于 Transformer 在蛋白质结构预测中应用的综述。
每个论点必须标注来源 [doc_id, offset]。
不要编造任何未在证据中出现的信息。

{evidence_text}"""
    }]
)
print(msg.content[0].text)


## 注意事项

- 所有引用必须来自 Sciverse 返回的真实 doc_id 和 offset，不要让 LLM 编造
- agentic-search 的 top_k 范围为 1–100，综述场景建议 top_k=20
- content 接口默认 limit=700 字符；如需更多上下文可传入更大值（如 2000–4096）
- 如需全文，可循环调用 content 并使用 next_offset 拼接
- 生产环境建议加 try/except 处理 404（文档无全文）和 429（限流）


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
